In [8]:
import os
import subprocess
from pathlib import Path
import pandas as pd
from typing import List, Optional

# =========================
# CONFIG (EDIT THESE PATHS)
# =========================
CLONE_ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Clone")
BOUNDARY_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\List_Boundary_Commit_Events.csv")  # <-- change to where your csv is
OUTDIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5")
OUTDIR.mkdir(parents=True, exist_ok=True)

WORKFLOW_PATHS = [".github/workflows", ".github/main.workflow"]

# Git executable autodetect
GIT_EXE_CANDIDATES = [
    "git",
    r"C:\Program Files\Git\bin\git.exe",
    r"C:\Program Files\Git\cmd\git.exe",
    r"C:\Program Files (x86)\Git\bin\git.exe",
    r"C:\Program Files (x86)\Git\cmd\git.exe",
]

def pick_git_exe() -> str:
    for exe in GIT_EXE_CANDIDATES:
        try:
            r = subprocess.run([exe, "--version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            if r.returncode == 0 and "git version" in (r.stdout.lower() + r.stderr.lower()):
                return exe
        except FileNotFoundError:
            continue
    raise FileNotFoundError("Could not find git.exe. Install Git for Windows or add it to PATH.")

GIT_EXE = pick_git_exe()
print("Using git:", GIT_EXE)

def run_git(repo_dir: Path, args: List[str]) -> subprocess.CompletedProcess:
    return subprocess.run(
        [GIT_EXE, "-C", str(repo_dir), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

def is_git_repo(repo_dir: Path) -> bool:
    r = run_git(repo_dir, ["rev-parse", "--is-inside-work-tree"])
    return r.returncode == 0 and r.stdout.strip() == "true"

def commit_exists(repo_dir: Path, sha: str) -> bool:
    r = run_git(repo_dir, ["cat-file", "-e", f"{sha}^{{commit}}"])
    return r.returncode == 0

def actions_present_at_commit(repo_dir: Path, sha: str) -> Optional[bool]:
    """
    True  => at commit sha, at least one Actions workflow path exists
    False => commit exists, but no workflow paths exist at that commit
    None  => repo/commit missing locally or other git error
    """
    if not is_git_repo(repo_dir):
        return None
    if not commit_exists(repo_dir, sha):
        return None

    r = run_git(repo_dir, ["ls-tree", "-r", "--name-only", sha, "--", *WORKFLOW_PATHS])
    if r.returncode != 0:
        return None
    return bool(r.stdout.strip())

def first_actions_commit(repo_dir: Path) -> Optional[str]:
    r = run_git(repo_dir, ["rev-list", "--all", "--reverse", "-n", "1", "--", *WORKFLOW_PATHS])
    sha = r.stdout.strip() if r.returncode == 0 else ""
    return sha or None

def commit_iso_date(repo_dir: Path, sha: str) -> Optional[str]:
    r = run_git(repo_dir, ["show", "-s", "--format=%cI", sha])
    if r.returncode != 0:
        return None
    d = r.stdout.strip()
    return d if d else None

# =========================
# LOAD + PICK FIRST START BOUNDARY PER REPO
# =========================
df = pd.read_csv(BOUNDARY_CSV)

# Ensure types / sort keys
df["is_episode_start_boundary"] = df["is_episode_start_boundary"].astype(int)
df["episode_index"] = df["episode_index"].astype(int)
df["event_date_utc"] = pd.to_datetime(df["event_date_utc"], errors="coerce")

starts = df[df["is_episode_start_boundary"] == 1].copy()

# "first instrumentation episode" = min episode_index, tie-break by earliest date
starts = starts.sort_values(["repo_name", "episode_index", "event_date_utc"])
first_start = starts.groupby("repo_name", as_index=False).first()

print("Repos with at least one start boundary:", first_start["repo_name"].nunique())

# =========================
# CHECK ACTIONS PRESENCE AT THAT START COMMIT
# =========================
records = []
missing_repo_dirs = 0

for _, row in first_start.iterrows():
    repo_name = row["repo_name"]
    repo_dir = CLONE_ROOT / repo_name
    sha = str(row["commit_sha"]).strip()

    rec = {
        "repo_name": repo_name,
        "repo_dir": str(repo_dir),
        "full_name": row.get("full_name", ""),
        "first_episode_index": int(row["episode_index"]),
        "first_start_env_style": row.get("env_style", ""),
        "first_start_event_type": row.get("event_type", ""),
        "first_start_event_date_utc": str(row["event_date_utc"]) if pd.notna(row["event_date_utc"]) else "",
        "first_start_commit_sha": sha,
        "repo_found_locally": repo_dir.exists() and repo_dir.is_dir(),
        "is_git_repo": False,
        "actions_present_at_first_start_boundary": "",
        "first_actions_commit_in_history": "",
        "first_actions_date_in_history": "",
        "error": "",
    }

    if not rec["repo_found_locally"]:
        missing_repo_dirs += 1
        rec["error"] = "REPO_DIR_NOT_FOUND"
        records.append(rec)
        continue

    rec["is_git_repo"] = is_git_repo(repo_dir)
    if not rec["is_git_repo"]:
        rec["error"] = "NOT_A_GIT_REPO"
        records.append(rec)
        continue

    present = actions_present_at_commit(repo_dir, sha)
    if present is None:
        rec["actions_present_at_first_start_boundary"] = ""
        rec["error"] = "START_SHA_NOT_FOUND_LOCALLY_OR_GIT_ERROR"
    else:
        rec["actions_present_at_first_start_boundary"] = bool(present)

    fac = first_actions_commit(repo_dir)
    rec["first_actions_commit_in_history"] = fac or ""
    rec["first_actions_date_in_history"] = commit_iso_date(repo_dir, fac) if fac else ""

    records.append(rec)

out = pd.DataFrame(records)

# =========================
# WRITE OUTPUTS
# =========================
all_path = OUTDIR / "repos_first_episode_start_has_actions.csv"
out.to_csv(all_path, index=False)

true_subset = out[out["actions_present_at_first_start_boundary"] == True].copy()
true_path = OUTDIR / "repos_first_episode_start_has_actions_TRUE.csv"
true_subset.to_csv(true_path, index=False)

print("\nDone.")
print("Missing repo dirs:", missing_repo_dirs)
print("Total repos checked:", len(out))
print("Actions present at FIRST start boundary:", (out["actions_present_at_first_start_boundary"] == True).sum())
print("Wrote:\n -", all_path, "\n -", true_path)


Using git: git
Repos with at least one start boundary: 399

Done.
Missing repo dirs: 1
Total repos checked: 399
Actions present at FIRST start boundary: 201
Wrote:
 - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\repos_first_episode_start_has_actions.csv 
 - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\5_RQ5\repos_first_episode_start_has_actions_TRUE.csv


In [11]:
# token check

In [13]:
from pathlib import Path
import requests
import re

ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

def load_env_tokens(env_path: Path):
    """
    Reads KEY=VALUE lines. Supports quoted values and ignores comments/blank lines.
    Returns dict of env vars found in file.
    """
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")

    env = {}
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        env[k] = v
    return env

env = load_env_tokens(ENV_PATH)

# Collect your four tokens
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
tokens = [(k, env.get(k, "").strip()) for k in token_keys]

print("Loaded token lengths:")
for k, t in tokens:
    print(k, len(t))

def test_token(token: str):
    """
    Tests token validity. Returns (ok: bool, status_code: int, login: str|None).
    """
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "Authorization": f"token {token}",  # <— use 'token' form (works well with PATs)
    }
    r = requests.get("https://api.github.com/user", headers=headers, timeout=30)
    if r.status_code == 200:
        return True, r.status_code, r.json().get("login")
    return False, r.status_code, None

working = []
for k, t in tokens:
    if not t:
        print(f"{k}: EMPTY")
        continue
    ok, status, login = test_token(t)
    print(f"{k}: status={status}, ok={ok}, login={login}")
    if ok:
        working.append((k, t, login))

if not working:
    raise RuntimeError("No working tokens found. 401 means invalid/expired token OR wrong host (Enterprise).")

# Pick the first working token
ACTIVE_TOKEN_KEY, ACTIVE_TOKEN, ACTIVE_LOGIN = working[0]
print("\nUsing:", ACTIVE_TOKEN_KEY, "as", ACTIVE_LOGIN)


Loaded token lengths:
GITHUB_TOKEN_1 40
GITHUB_TOKEN_2 40
GITHUB_TOKEN_3 40
GITHUB_TOKEN_4 40
GITHUB_TOKEN_1: status=200, ok=True, login=BehnamParsa4
GITHUB_TOKEN_2: status=200, ok=True, login=BehnamParsa2
GITHUB_TOKEN_3: status=200, ok=True, login=BehnamAcc6
GITHUB_TOKEN_4: status=200, ok=True, login=BehnamParsa5

Using: GITHUB_TOKEN_1 as BehnamParsa4


In [4]:
# Step 1 by working token


In [10]:
import re
import time
from pathlib import Path
import pandas as pd
import requests

# =========================
# PATHS / CONFIG
# =========================
INPUT_TRUE = Path(r"D:\5_RQ5\repos_first_episode_start_has_actions_TRUE.csv")
OUTDIR     = Path(r"D:\5_RQ5")
ENV_PATH   = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

OUTDIR.mkdir(parents=True, exist_ok=True)
OUT_ALL = OUTDIR / "gha_run_availability_screen.csv"
OUT_OK  = OUTDIR / "gha_metrics_eligible_repos.csv"

# =========================
# LOAD TOKEN_3
# =========================
def load_env_tokens(env_path: Path):
    env = {}
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env_tokens(ENV_PATH)
TOKEN = env.get("GITHUB_TOKEN_3", "").strip()
if not TOKEN:
    raise ValueError("GITHUB_TOKEN_3 not found or empty in All_Tokens.env")

# =========================
# HELPERS
# =========================
def repo_slug(row) -> str:
    # Prefer full_name if present
    fn = str(row.get("full_name", "")).strip()
    if fn and "/" in fn and " " not in fn:
        return fn

    # Fallback: repo_name like Owner__Repo
    rn = str(row.get("repo_name", "")).strip()
    m = re.match(r"^([^_]+)__([^_]+)$", rn)
    if m:
        return f"{m.group(1)}/{m.group(2)}"
    return ""

def gh_get(url, params=None):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "Authorization": f"token {TOKEN}",
    }
    return requests.get(url, headers=headers, params=params, timeout=30)

# Sanity check
t = gh_get("https://api.github.com/user")
print("Token sanity check:", t.status_code, (t.json().get("login") if t.status_code == 200 else t.text[:200]))
if t.status_code != 200:
    raise RuntimeError("Token is not working in this notebook session.")

# =========================
# LOAD INPUT
# =========================
df = pd.read_csv(INPUT_TRUE)

if "first_start_event_date_utc" not in df.columns:
    raise ValueError("Missing column 'first_start_event_date_utc' in repos_first_episode_start_has_actions_TRUE.csv")

records = []
n = len(df)

for i, row in df.iterrows():
    slug = repo_slug(row)
    start_dt_raw = str(row["first_start_event_date_utc"]).strip()
    start_date = start_dt_raw.split(" ")[0].split("T")[0]  # YYYY-MM-DD

    rec = row.to_dict()
    rec["owner_repo"] = slug
    rec["start_date_yyyy_mm_dd"] = start_date
    rec["api_status"] = ""
    rec["api_note"] = ""
    rec["runs_total_since_start"] = ""  # keep raw; we'll make numeric later

    if not slug or not start_date or start_date.lower() == "nan":
        rec["api_status"] = "SKIP_MISSING_SLUG_OR_DATE"
        rec["runs_total_since_start"] = ""
        records.append(rec)
        continue

    url = f"https://api.github.com/repos/{slug}/actions/runs"
    params = {"per_page": 1, "created": f">={start_date}"}

    r = gh_get(url, params=params)
    rec["api_status"] = r.status_code

    # helpful for debugging rate limits
    rec["rate_remaining"] = r.headers.get("X-RateLimit-Remaining", "")
    rec["rate_reset"] = r.headers.get("X-RateLimit-Reset", "")

    if r.status_code == 200:
        j = r.json()
        rec["runs_total_since_start"] = int(j.get("total_count", 0))
    else:
        # Always set something (avoid blank-int crash later)
        rec["runs_total_since_start"] = ""
        try:
            if "application/json" in r.headers.get("content-type", ""):
                rec["api_note"] = r.json().get("message", "")[:300]
            else:
                rec["api_note"] = r.text[:300]
        except Exception:
            rec["api_note"] = "Could not parse error body"

    records.append(rec)

    # Gentle throttling
    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{n} ...")
        time.sleep(1.5)

out = pd.DataFrame(records)

# SAFE conversion: blanks/strings -> NaN
out["runs_total_since_start_num"] = pd.to_numeric(out["runs_total_since_start"], errors="coerce")

out.to_csv(OUT_ALL, index=False)

eligible = out[(out["api_status"] == 200) & (out["runs_total_since_start_num"] > 0)].copy()
eligible.to_csv(OUT_OK, index=False)

print("\nWrote:", OUT_ALL)
print("Wrote:", OUT_OK)
print("Eligible repos:", len(eligible), "out of", len(out))
print("\nStatus breakdown:")
print(out["api_status"].value_counts().head(15))

# Show top failing reasons (optional)
non200 = out[out["api_status"] != 200][["owner_repo","api_status","api_note","rate_remaining"]].head(20)
print("\nSample non-200 rows:")
print(non200.to_string(index=False))


Token sanity check: 200 BehnamAcc6
Processed 50/201 ...
Processed 100/201 ...
Processed 150/201 ...

Wrote: D:\5_RQ5\gha_run_availability_screen.csv
Wrote: D:\5_RQ5\gha_metrics_eligible_repos.csv
Eligible repos: 145 out of 201

Status breakdown:
api_status
200                          191
SKIP_MISSING_SLUG_OR_DATE     10
Name: count, dtype: int64

Sample non-200 rows:
owner_repo                api_status api_note rate_remaining
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MISSING_SLUG_OR_DATE                     NaN
           SKIP_MIS

In [ ]:
#Step 2A: Collect runs_index.csv (repo → runs between start date and cutoff)

In [ ]:
import re
import time
import itertools
from pathlib import Path
import pandas as pd
import requests

BASE = Path(r"D:\5_RQ5")
ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

INPUT = BASE / "gha_metrics_eligible_repos_Sample.csv"
OUTPUT = BASE / "runs_index.csv"

CUTOFF_DATE = "2025-08-10"  # inclusive range end (your cutoff)

# -------------------------
# Token loading + rotation
# -------------------------
def load_env_tokens(env_path: Path):
    env = {}
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env_tokens(ENV_PATH)
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
raw_tokens = [(k, env.get(k, "").strip()) for k in token_keys if env.get(k, "").strip()]

def token_ok(tok: str):
    h = {"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28","Authorization":f"token {tok}"}
    r = requests.get("https://api.github.com/user", headers=h, timeout=30)
    return (r.status_code == 200, (r.json().get("login") if r.status_code == 200 else None), r.status_code)

working = []
for k, t in raw_tokens:
    ok, login, status = token_ok(t)
    print(f"{k}: status={status}, ok={ok}, login={login}")
    if ok:
        working.append((k, t, login))

if not working:
    raise RuntimeError("No working tokens. Fix token permissions or file path.")

print("Working tokens:", [k for k,_,_ in working])
TOKEN_CYCLE = itertools.cycle([t for _, t, _ in working])

def gh_get(url, params=None, max_retries=6):
    """GET with token rotation + rate-limit sleep."""
    last = None
    for attempt in range(max_retries):
        tok = next(TOKEN_CYCLE)
        headers = {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Authorization": f"token {tok}",
        }
        r = requests.get(url, headers=headers, params=params, timeout=60)
        last = r

        # OK
        if r.status_code == 200:
            return r

        # Rate limit / abuse detection: sleep until reset when possible
        rem = r.headers.get("X-RateLimit-Remaining")
        reset = r.headers.get("X-RateLimit-Reset")
        msg = ""
        try:
            if "application/json" in r.headers.get("content-type", ""):
                msg = r.json().get("message", "")
        except Exception:
            pass

        if r.status_code == 403 and (rem == "0" or "rate limit" in msg.lower()):
            if reset and reset.isdigit():
                sleep_s = max(10, int(reset) - int(time.time()) + 10)
                print(f"[rate limit] sleeping {sleep_s}s...")
                time.sleep(sleep_s)
                continue

        # transient
        if r.status_code in (500, 502, 503, 504):
            time.sleep(2 + attempt * 2)
            continue

        # other errors: return as-is
        return r

    return last

# -------------------------
# Load eligible repos
# -------------------------
df = pd.read_csv(INPUT)

# Expect these from your Step-1 output
if "owner_repo" not in df.columns:
    raise ValueError("Missing column 'owner_repo' in gha_metrics_eligible_repos.csv")
if "start_date_yyyy_mm_dd" not in df.columns:
    raise ValueError("Missing column 'start_date_yyyy_mm_dd' in gha_metrics_eligible_repos.csv")

# Resume support: if OUTPUT exists, skip runs already collected
existing_run_ids = set()
if OUTPUT.exists():
    try:
        prev = pd.read_csv(OUTPUT, usecols=["run_id"])
        existing_run_ids = set(prev["run_id"].astype(str).tolist())
        print(f"Resume: found {len(existing_run_ids)} existing run_ids in {OUTPUT.name}")
    except Exception:
        pass

out_rows = []
write_header = not OUTPUT.exists()

def flush(rows):
    if not rows:
        return
    pd.DataFrame(rows).to_csv(OUTPUT, mode="a", index=False, header=write_header if OUTPUT.stat().st_size == 0 else False)
    rows.clear()

# -------------------------
# Fetch runs per repo
# -------------------------
print(f"Collecting runs into: {OUTPUT}")

for idx, row in df.iterrows():
    owner_repo = str(row["owner_repo"]).strip()
    start_date = str(row["start_date_yyyy_mm_dd"]).strip()

    if not owner_repo or start_date.lower() == "nan":
        continue

    # Created range syntax: START..END
    created_range = f"{start_date}..{CUTOFF_DATE}"

    url = f"https://api.github.com/repos/{owner_repo}/actions/runs"
    params = {"per_page": 100, "page": 1, "created": created_range}

    while True:
        r = gh_get(url, params=params)
        if r is None:
            break

        if r.status_code != 200:
            # record repo-level failure row (optional)
            print(f"[WARN] {owner_repo} page={params['page']} status={r.status_code}")
            break

        data = r.json()
        runs = data.get("workflow_runs", [])

        for run in runs:
            run_id = str(run.get("id", ""))
            if not run_id or run_id in existing_run_ids:
                continue

            out_rows.append({
                "owner_repo": owner_repo,
                "run_id": run_id,
                "run_number": run.get("run_number"),
                "run_attempt": run.get("run_attempt"),
                "event": run.get("event"),
                "status": run.get("status"),
                "conclusion": run.get("conclusion"),
                "created_at": run.get("created_at"),
                "updated_at": run.get("updated_at"),
                "run_started_at": run.get("run_started_at"),
                "head_sha": run.get("head_sha"),
                "head_branch": run.get("head_branch"),
                "workflow_id": run.get("workflow_id"),
                "path": run.get("path"),
                "html_url": run.get("html_url"),
            })
            existing_run_ids.add(run_id)

        # pagination: stop when less than 100 returned
        if len(runs) < 100:
            break
        params["page"] += 1

        # flush periodically
        if len(out_rows) >= 5000:
            flush(out_rows)

    if (idx + 1) % 10 == 0:
        print(f"Processed repos {idx+1}/{len(df)}")

flush(out_rows)
print("Done. Wrote:", OUTPUT)


In [ ]:
#Step 2B: Collect jobs_steps.csv (run_id → jobs + steps timestamps)

In [ ]:
import time
import itertools
from pathlib import Path
import pandas as pd
import requests
import json

BASE = Path(r"D:\5_RQ5")
ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

INPUT = BASE / "runs_index.csv"
OUTPUT = BASE / "jobs_steps.csv"

# -------------------------
# Token loading + rotation
# -------------------------
def load_env_tokens(env_path: Path):
    env = {}
    for line in env_path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env

env = load_env_tokens(ENV_PATH)
token_keys = [f"GITHUB_TOKEN_{i}" for i in range(1, 5)]
tokens = [env.get(k, "").strip() for k in token_keys if env.get(k, "").strip()]

def token_ok(tok: str):
    h = {"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28","Authorization":f"token {tok}"}
    r = requests.get("https://api.github.com/user", headers=h, timeout=30)
    return r.status_code == 200

working_tokens = [t for t in tokens if token_ok(t)]
if not working_tokens:
    raise RuntimeError("No working tokens for Pass 2.")
TOKEN_CYCLE = itertools.cycle(working_tokens)

def gh_get(url, params=None, max_retries=6):
    last = None
    for attempt in range(max_retries):
        tok = next(TOKEN_CYCLE)
        headers = {
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Authorization": f"token {tok}",
        }
        r = requests.get(url, headers=headers, params=params, timeout=60)
        last = r

        if r.status_code == 200:
            return r

        rem = r.headers.get("X-RateLimit-Remaining")
        reset = r.headers.get("X-RateLimit-Reset")
        msg = ""
        try:
            if "application/json" in r.headers.get("content-type", ""):
                msg = r.json().get("message", "")
        except Exception:
            pass

        if r.status_code == 403 and (rem == "0" or "rate limit" in msg.lower()):
            if reset and reset.isdigit():
                sleep_s = max(10, int(reset) - int(time.time()) + 10)
                print(f"[rate limit] sleeping {sleep_s}s...")
                time.sleep(sleep_s)
                continue

        if r.status_code in (500, 502, 503, 504):
            time.sleep(2 + attempt * 2)
            continue

        return r

    return last

# -------------------------
# Load runs
# -------------------------
runs = pd.read_csv(INPUT, dtype={"run_id": str})
print("Runs loaded:", len(runs))

# Resume: skip already processed run_ids
processed = set()
if OUTPUT.exists():
    try:
        prev = pd.read_csv(OUTPUT, usecols=["run_id"], dtype={"run_id": str})
        processed = set(prev["run_id"].astype(str).tolist())
        print(f"Resume: already processed {len(processed)} run_ids")
    except Exception:
        pass

out_rows = []
def flush(rows):
    if not rows:
        return
    header = not OUTPUT.exists() or OUTPUT.stat().st_size == 0
    pd.DataFrame(rows).to_csv(OUTPUT, mode="a", index=False, header=header)
    rows.clear()

# -------------------------
# Fetch jobs+steps for each run
# -------------------------
for i, rrow in runs.iterrows():
    owner_repo = str(rrow["owner_repo"]).strip()
    run_id = str(rrow["run_id"]).strip()
    if not owner_repo or not run_id or run_id in processed:
        continue

    url = f"https://api.github.com/repos/{owner_repo}/actions/runs/{run_id}/jobs"
    params = {"per_page": 100, "page": 1}

    while True:
        resp = gh_get(url, params=params)
        if resp is None or resp.status_code != 200:
            print(f"[WARN] jobs failed {owner_repo} run={run_id} status={None if resp is None else resp.status_code}")
            break

        data = resp.json()
        jobs = data.get("jobs", [])

        for job in jobs:
            job_id = job.get("id")
            job_name = job.get("name")
            job_started = job.get("started_at")
            job_completed = job.get("completed_at")
            job_conclusion = job.get("conclusion")
            job_status = job.get("status")
            runner_name = job.get("runner_name")
            runner_group = job.get("runner_group_name")

            # NEW: job labels (useful for self-hosted vs ubuntu-latest etc.)
            job_labels = job.get("labels", None)
            if isinstance(job_labels, list):
                job_labels = ",".join([str(x) for x in job_labels])
            elif job_labels is None:
                job_labels = ""

            steps = job.get("steps", []) or []
            for step in steps:
                out_rows.append({
                    "owner_repo": owner_repo,
                    "run_id": run_id,

                    "job_id": job_id,
                    "job_name": job_name,
                    "job_status": job_status,
                    "job_conclusion": job_conclusion,
                    "job_started_at": job_started,
                    "job_completed_at": job_completed,
                    "runner_name": runner_name,
                    "runner_group_name": runner_group,
                    "job_labels": job_labels,

                    "step_name": step.get("name"),
                    "step_status": step.get("status"),
                    "step_conclusion": step.get("conclusion"),
                    "step_number": step.get("number"),
                    "step_started_at": step.get("started_at"),
                    "step_completed_at": step.get("completed_at"),
                })

        if len(jobs) < 100:
            break
        params["page"] += 1

        if len(out_rows) >= 20000:
            flush(out_rows)

    processed.add(run_id)

    if (i + 1) % 200 == 0:
        print(f"Processed runs {i+1}/{len(runs)}")
        flush(out_rows)

flush(out_rows)
print("Done. Wrote:", OUTPUT)


In [ ]:
#Step 2C: Identify instrumentation steps + compute durations + attach style label

In [ ]:
import re
import pandas as pd
from pathlib import Path

BASE = Path(r"D:\5_RQ5")
JOBS_STEPS = BASE / "jobs_steps.csv"
RUNS_INDEX = BASE / "runs_index.csv"
BOUNDARY   = BASE / "List_Boundary_Commit_Events.csv"
GMD_SCAN_CSV = BASE / "1_Gradle_GMD_Instru.csv"  # optional

OUT_STEP_LEVEL = BASE / "instrumentation_steps_labeled.csv"
OUT_RUN_LEVEL  = BASE / "run_level_perf_summary.csv"
OUT_STYLE_SUM  = BASE / "summary_by_env_style_detected.csv"
OUT_EXIT_SUM   = BASE / "repo_instr_exit_summary.csv"

# -------------------------
# Parameters for "exit" detection
# -------------------------
CUTOFF_DATE = pd.to_datetime("2025-08-10", utc=True)   # your cutoff
EXIT_GAP_DAYS = 120                                   # no instr for >=120 days while GHA still runs => likely exit
ACTIVITY_WINDOW_DAYS = 180                            # require repo still active on GHA near cutoff (to avoid inactive repos)

# -------------------------
# Load data
# -------------------------
steps = pd.read_csv(JOBS_STEPS, dtype={"run_id": str})
runs  = pd.read_csv(RUNS_INDEX, dtype={"run_id": str})
b     = pd.read_csv(BOUNDARY)

# Normalize datetimes
for col in ["step_started_at","step_completed_at","job_started_at","job_completed_at"]:
    if col in steps.columns:
        steps[col] = pd.to_datetime(steps[col], errors="coerce", utc=True)

runs["created_at"] = pd.to_datetime(runs["created_at"], errors="coerce", utc=True)

# Join run info into steps
steps = steps.merge(
    runs[["owner_repo","run_id","created_at","head_sha","path","conclusion","run_attempt","event"]],
    on=["owner_repo","run_id"],
    how="left"
)

# Step duration
steps["step_duration_sec"] = (steps["step_completed_at"] - steps["step_started_at"]).dt.total_seconds()

# Safe strings
steps["step_name_str"] = steps["step_name"].fillna("").astype(str)
steps["job_name_str"]  = steps["job_name"].fillna("").astype(str)

# -------------------------
# Load GMD config signal (repo-level, optional)
# -------------------------
def full_name_to_owner_repo(full_name: str) -> str:
    s = (full_name or "").strip()
    if "/" in s:
        return s
    if "." in s:
        a, b = s.split(".", 1)
        return f"{a}/{b}"
    if "__" in s:
        return s.replace("__", "/")
    return ""

gmd_repo_has_config = {}
if GMD_SCAN_CSV.exists():
    gmd = pd.read_csv(GMD_SCAN_CSV)
    if "full_name" in gmd.columns and "gmd_config_signal" in gmd.columns:
        gmd["owner_repo"] = gmd["full_name"].astype(str).apply(full_name_to_owner_repo)
        gmd_repo_has_config = (gmd.groupby("owner_repo")["gmd_config_signal"].max().to_dict())
        print("Loaded GMD config signals for repos:", len(gmd_repo_has_config))
    else:
        print("[WARN] GMD scan csv missing required columns: full_name, gmd_config_signal.")
else:
    print("[INFO] No GMD scan CSV found. Will rely on step-name detection for GMD.")

steps["repo_has_gmd_config"] = steps["owner_repo"].map(gmd_repo_has_config).fillna(False).astype(bool)

# -------------------------
# DETECTORS (Invocation + Provisioning)
# -------------------------
INSTR_PATTERNS = {
    "Gradle_Connected": r"(connectedandroidtest|connectedcheck|devicecheck|alldevicechecks|\bandroidtest\b)",
    "Gradle_GMD": r"(manageddevice|managed device|\bgmd\b|gradle managed devices)",
    "ADB": r"(am\s+instrument|\badb\b.*\binstrument\b)",
    "Firebase_Test_Lab": r"(firebase test lab|\bgcloud\b.*firebase\s+test\s+android\s+run|\bftl\b|\bflank\b)",
    "AppCenter": r"(appcenter\s+test\s+run\s+android|app center)",
    "BrowserStack": r"(browserstack|\bbstack\b)",
    "SauceLabs": r"(saucectl|saucelabs)",
    "EmulatorWTF": r"(emulator\.wtf|emulator-wtf)",
    "Maestro_Cloud": r"(maestro\s+cloud)",
    "Marathon": r"\bmarathon\b",
    "Spoon": r"\bspoon\b",
}
INSTR_RE = {k: re.compile(v, re.I) for k,v in INSTR_PATTERNS.items()}

PROVISION_PATTERNS = {
    "Emu_Start": r"(emulator\s+-avd|avdmanager|android-wait-for-emulator|start-emulator|no-window|no-boot-anim)",
    "Emu_Wait": r"(adb\s+wait[- ]?for[- ]?device|wait for device|boot completed|sys\.boot_completed)",
    "Emu_Community_Action": r"(android-emulator-runner|reactivecircus|malinskiy|emulator-run-cmd)",
    "SDK_Setup": r"(sdkmanager|system-images;android-|platform-tools|cmdline-tools)",
}
PROV_RE = {k: re.compile(v, re.I) for k,v in PROVISION_PATTERNS.items()}

def first_match_label(text: str, regex_map):
    for lbl, rx in regex_map.items():
        if rx.search(text):
            return lbl
    return ""

def all_match_labels(text: str, regex_map):
    return [lbl for lbl, rx in regex_map.items() if rx.search(text)]

steps["instr_label"] = steps["step_name_str"].apply(lambda s: first_match_label(s, INSTR_RE))
steps["prov_labels"] = steps["step_name_str"].apply(lambda s: ",".join(all_match_labels(s, PROV_RE)))

# -------------------------
# Execution environment style (detected)
# -------------------------
EMU_GMD_LABEL = "Emu_GMD"

def detect_env_style_step(row) -> str:
    s = (row["step_name_str"] + " " + row["job_name_str"]).lower()

    # 3P labs
    if any(x in s for x in ["firebase", "flank", "appcenter", "browserstack", "bstack", "saucectl", "emulator.wtf", "maestro cloud"]):
        return "Third-Party"

    # explicit GMD in text
    if any(x in s for x in ["manageddevice", "managed device", "gradle managed", " gmd "]):
        return EMU_GMD_LABEL

    # emulator community actions
    if any(x in s for x in ["android-emulator-runner", "reactivecircus", "malinskiy", "emulator-run-cmd"]):
        return "Emu_Community"

    # custom emulator
    if any(x in s for x in ["emulator -avd", "avdmanager", "adb wait-for-device", "android-wait-for-emulator", "start-emulator"]):
        return "Emu_Custom"

    return ""

steps["env_style_detected_step"] = steps.apply(detect_env_style_step, axis=1)

def detect_provider(text: str) -> str:
    t = (text or "").lower()
    if "firebase" in t or "flank" in t or "gcloud" in t or "ftl" in t:
        return "Firebase_Test_Lab"
    if "appcenter" in t or "app center" in t:
        return "AppCenter"
    if "browserstack" in t or "bstack" in t:
        return "BrowserStack"
    if "saucectl" in t or "saucelabs" in t:
        return "SauceLabs"
    if "emulator.wtf" in t or "emulator-wtf" in t:
        return "Emulator.wtf"
    if "maestro cloud" in t:
        return "Maestro_Cloud"
    return ""

steps["third_party_provider_step"] = steps["step_name_str"].apply(detect_provider)

# -------------------------
# Attach env_style from boundary episodes (RQ2 linkage)
# -------------------------
b["is_episode_start_boundary"] = b["is_episode_start_boundary"].astype(int)
b["episode_index"] = b["episode_index"].astype(int)
b["event_date_utc"] = pd.to_datetime(b["event_date_utc"], errors="coerce", utc=True)

starts = b[b["is_episode_start_boundary"] == 1].copy()
starts = starts.sort_values(["repo_name","event_date_utc"])

def boundary_to_owner_repo(row):
    fn = str(row.get("full_name","")).strip()
    if fn and "/" in fn:
        return fn
    rn = str(row.get("repo_name","")).strip()
    if "__" in rn:
        return rn.replace("__", "/")
    if "." in rn:
        a, bb = rn.split(".", 1)
        return f"{a}/{bb}"
    return ""

starts["owner_repo"] = starts.apply(boundary_to_owner_repo, axis=1)
starts = starts[starts["owner_repo"] != ""]

starts_grp = {k: g[["event_date_utc","env_style","episode_index"]].sort_values("event_date_utc")
              for k, g in starts.groupby("owner_repo")}

def label_style(owner_repo, created_at):
    g = starts_grp.get(owner_repo)
    if g is None or pd.isna(created_at):
        return pd.Series({"env_style_rq2": None, "episode_index_rq2": None})
    g2 = g[g["event_date_utc"] <= created_at]
    if g2.empty:
        return pd.Series({"env_style_rq2": None, "episode_index_rq2": None})
    last = g2.iloc[-1]
    return pd.Series({"env_style_rq2": last["env_style"], "episode_index_rq2": int(last["episode_index"])})

rq2_labels = steps.apply(lambda r: label_style(r["owner_repo"], r["created_at"]), axis=1)
steps = pd.concat([steps, rq2_labels], axis=1)

# -------------------------
# STEP-LEVEL output: instrumentation steps only
# -------------------------
instr_steps = steps[
    (steps["instr_label"] != "") &
    (steps["step_duration_sec"].notna()) &
    (steps["step_duration_sec"] >= 0)
].copy()

instr_steps.to_csv(OUT_STEP_LEVEL, index=False)
print("Wrote:", OUT_STEP_LEVEL, "rows:", len(instr_steps))

# -------------------------
# RUN/JOB-LEVEL perf summary
# -------------------------
prov = steps.copy()
prov["is_prov"] = prov["prov_labels"].astype(str).str.len() > 0
prov_sum = (prov[prov["is_prov"] & prov["step_duration_sec"].notna()]
            .groupby(["owner_repo","run_id","job_id"], as_index=False)
            .agg(provisioning_sec=("step_duration_sec","sum")))

instr_sum = (instr_steps.groupby(["owner_repo","run_id","job_id"], as_index=False)
             .agg(instrumentation_sec=("step_duration_sec","sum"),
                  instr_steps_count=("step_duration_sec","size")))

job_level = steps.drop_duplicates(subset=["owner_repo","run_id","job_id"])[
    ["owner_repo","run_id","job_id","job_name","job_started_at","job_completed_at","created_at",
     "run_attempt","env_style_rq2","episode_index_rq2","repo_has_gmd_config"]
].copy()

job_level["job_duration_sec"] = (job_level["job_completed_at"] - job_level["job_started_at"]).dt.total_seconds()
job_level["queue_time_sec"] = (job_level["job_started_at"] - job_level["created_at"]).dt.total_seconds()

STYLE_PRIORITY = ["Third-Party", EMU_GMD_LABEL, "Emu_Community", "Emu_Custom", "Real Device"]
def choose_style(styles):
    styles = [s for s in styles if s]
    if not styles:
        return ""
    for p in STYLE_PRIORITY:
        if p in styles:
            return p
    return styles[0]

job_style = (steps.groupby(["owner_repo","run_id","job_id"], as_index=False)
             .agg(env_style_detected=("env_style_detected_step", lambda s: choose_style(list(set([x for x in s if x])))),
                  third_party_provider=("third_party_provider_step", lambda s: choose_style(list(set([x for x in s if x]))))))

run_job = job_level.merge(instr_sum, on=["owner_repo","run_id","job_id"], how="left")
run_job = run_job.merge(prov_sum, on=["owner_repo","run_id","job_id"], how="left")
run_job = run_job.merge(job_style, on=["owner_repo","run_id","job_id"], how="left")

run_job["instrumentation_sec"] = run_job["instrumentation_sec"].fillna(0)
run_job["provisioning_sec"] = run_job["provisioning_sec"].fillna(0)

# GMD config fallback
mask_gmd_fallback = (
    (run_job["env_style_detected"].astype(str).str.len() == 0) &
    (run_job["repo_has_gmd_config"] == True) &
    (run_job["instrumentation_sec"] > 0) &
    (run_job["third_party_provider"].astype(str).str.len() == 0)
)
run_job.loc[mask_gmd_fallback, "env_style_detected"] = EMU_GMD_LABEL

run_job.to_csv(OUT_RUN_LEVEL, index=False)
print("Wrote:", OUT_RUN_LEVEL, "rows:", len(run_job))

style_summary = (run_job[run_job["env_style_detected"].astype(str).str.len() > 0]
                 .groupby("env_style_detected", as_index=False)
                 .agg(
                     n_jobs=("job_id","count"),
                     median_job_sec=("job_duration_sec","median"),
                     median_queue_sec=("queue_time_sec","median"),
                     median_prov_sec=("provisioning_sec","median"),
                     median_instr_sec=("instrumentation_sec","median"),
                     p90_instr_sec=("instrumentation_sec", lambda s: s.quantile(0.90)),
                 )
                 .sort_values("n_jobs", ascending=False))

style_summary.to_csv(OUT_STYLE_SUM, index=False)
print("Wrote:", OUT_STYLE_SUM)
print(style_summary.head(20))

# -------------------------
# ✅ NEW: Repo-level "GHA instrumentation exit" detection
# -------------------------
# last instrumentation timestamp per repo
repo_last_instr = (instr_steps.groupby("owner_repo", as_index=False)
                   .agg(last_instr_at=("step_completed_at","max"),
                        first_instr_at=("step_started_at","min"),
                        instr_step_rows=("run_id","count")))

# last GHA run timestamp per repo (any workflow)
repo_last_run = (runs.groupby("owner_repo", as_index=False)
                 .agg(last_gha_run_at=("created_at","max"),
                      first_gha_run_at=("created_at","min"),
                      gha_run_count=("run_id","count")))

repo_exit = repo_last_run.merge(repo_last_instr, on="owner_repo", how="left")

# runs after last instr (key evidence for “moved away but GHA still running”)
# (Only compute where last_instr_at exists)
runs_min = runs[["owner_repo","created_at","run_id"]].dropna(subset=["created_at"]).copy()
repo_exit["runs_after_last_instr"] = 0

tmp = repo_exit.dropna(subset=["last_instr_at"])[["owner_repo","last_instr_at"]].copy()
if not tmp.empty:
    merged = runs_min.merge(tmp, on="owner_repo", how="inner")
    merged["after"] = merged["created_at"] > merged["last_instr_at"]
    after_counts = (merged[merged["after"]]
                    .groupby("owner_repo")["run_id"]
                    .count()
                    .to_dict())
    repo_exit["runs_after_last_instr"] = repo_exit["owner_repo"].map(after_counts).fillna(0).astype(int)

# define "active near cutoff"
active_threshold = CUTOFF_DATE - pd.Timedelta(days=ACTIVITY_WINDOW_DAYS)
repo_exit["gha_active_near_cutoff"] = repo_exit["last_gha_run_at"] >= active_threshold

# define "instr missing long enough"
repo_exit["days_since_last_instr"] = (CUTOFF_DATE - repo_exit["last_instr_at"]).dt.days
repo_exit["days_between_last_instr_and_last_run"] = (repo_exit["last_gha_run_at"] - repo_exit["last_instr_at"]).dt.days

# exit logic:
# - has instr at some point
# - still active on GHA near cutoff
# - no instr for >= EXIT_GAP_DAYS
# - AND there are GHA runs after last instr
repo_exit["gha_instr_exit_detected"] = (
    repo_exit["last_instr_at"].notna() &
    repo_exit["gha_active_near_cutoff"] &
    (repo_exit["days_since_last_instr"] >= EXIT_GAP_DAYS) &
    (repo_exit["runs_after_last_instr"] > 0)
)

# distinguish “inactive” (repo just stopped using GHA entirely)
repo_exit["gha_inactive_after_instr"] = (
    repo_exit["last_instr_at"].notna() &
    (~repo_exit["gha_active_near_cutoff"])
)

# note column: human-readable
def exit_note(row):
    if row["gha_instr_exit_detected"]:
        return "LIKELY_EXIT: GHA still running but no instrumentation observed"
    if row["gha_inactive_after_instr"]:
        return "INACTIVE: no recent GHA activity; cannot infer migration"
    if pd.isna(row["last_instr_at"]):
        return "NO_INSTR_OBSERVED_IN_GHA_METRICS"
    return "CONTINUES_OR_RECENT"

repo_exit["exit_note"] = repo_exit.apply(exit_note, axis=1)

repo_exit.to_csv(OUT_EXIT_SUM, index=False)
print("Wrote:", OUT_EXIT_SUM)
print("Exit detected repos:", int(repo_exit["gha_instr_exit_detected"].sum()), "out of", len(repo_exit))
